# QFT-Graph: A-5 gauge-augmentation experiment (Colab GPU runner)

Runs the committed CLI script `scripts/train_u1.py` with the A-5 flags
(`--gauge_augment`, `--n_train`) on the L=8 beta=2 ensemble (plan ground
rule 4: this notebook is only a launcher). Every run writes
`results/<run_id>.json` + a checkpoint under `experiments/runs/u1/`. The
driver **skips runs whose results JSON already exists** — after a
disconnect just re-run the cell.

**The A-5 question** (plan §3): Variants A/B sit at exact chance on all
gauge-invariant targets (the A-4 null) while the invariant-input oracle C
is at ceiling. Does on-the-fly random-gauge-transform augmentation drive
A/B's *learned* invariance (and accuracy) toward the C ceiling, and how
does that depend on training-set size? Protocol v2 throughout; the
augmentation draws a fresh gauge copy of every train config at every
access (labels are exactly gauge-invariant and reused; val/test stay
clean).

Run-id format now carries the train-size tag:
`prefix_variant_u1_L8_beta2_H64_B3_n{n_train}_seed{s}`.

**Intentionally absent runs:** the full-size (n=3200) no-augmentation
arms are byte-for-byte the a4null (A/B) and a4C (C) protocol runs, so
they are NOT repeated here — `scripts/plot_a5_curves.py` merges those
records into the curves.

| Cell | Experiment | Runs | est. GPU time* |
|---|---|---|---|
| F1a | A/B **augmented**, n_train in {50..3200} x 3 seeds | 42 | ~2.5-4 h |
| F1b | A/B baseline (no aug), n_train < 3200 x 3 seeds | 36 | ~1-2 h |
| F1c | C reference (no aug), n_train < 3200 x 3 seeds | 18 | ~0.5-1 h |
| F2 | (optional) volume check: A/B augmented @ L16 beta2, full size | 6 | ~1-2 h |
| F3 | eps_gauge on every new checkpoint (CPU-bound, or on laptop) | 102 | ~3-6 h CPU |

*Anchors: a4null full-size runs took ~17 min (A) / ~7 min (B, C) per seed
on Colab; run time scales roughly with n_train (constant 150 epochs), and
augmentation adds graph rebuilds (~0.6 ms/graph) in the DataLoader.
Sections are independent and resumable — split across sessions freely.


In [ ]:
import os, sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
assert IN_COLAB, 'This runner is meant for Colab (use scripts/train_u1.py locally)'

from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'
os.chdir(PROJECT_ROOT)

!pip install -q torch-geometric omegaconf h5py

# Make qft_graph importable for the CLI invocations below
os.environ['PYTHONPATH'] = os.path.join(PROJECT_ROOT, 'src')

import torch
print('CUDA available:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


## Physics gate (ground rule 1)

No training run launches until the exact-value tests pass **in this
environment** — Colab's torch/PyG versions differ from the laptop's.
The A-5 oracles are part of the suite: Variant C eps_gauge is exactly
zero through the real model path, gauge copies preserve action/W/Q to
1e-12, and the augmentation wrapper keeps labels while changing A/B
features. ~2-5 min. If anything fails, stop and report; do not weaken
tests.


In [ ]:
!python -m pytest tests/ -q


## Run driver

Builds CLI commands for `scripts/train_u1.py`, skips completed runs
(matching the script's `run_id` format exactly, including the `_n{n}`
train-size tag), streams output, and continues past failures.


In [ ]:
import subprocess, sys
from pathlib import Path

DATA = lambda L, beta: f'data/u1_configs/u1_L{L}_beta{beta:g}.h5'
SIZES = [50, 100, 200, 400, 800, 1600, 3200]

def run_id(r):
    # Must mirror scripts/train_u1.py exactly (prefix_variant_stem_H_B[_n]_seed)
    stem = Path(r['data']).stem
    ntag = f"_n{r['n_train']}" if r.get('n_train') is not None else ''
    return (f"{r['prefix']}_{r['variant']}_{stem}"
            f"_H{r.get('H', 64)}_B{r.get('B', 3)}{ntag}_seed{r['seed']}")

def launch(runs):
    failures = []
    for i, r in enumerate(runs, 1):
        rid = run_id(r)
        if (Path('results') / f'{rid}.json').exists():
            print(f'[{i}/{len(runs)}] SKIP (done): {rid}')
            continue
        # Argument list (no shell): immune to spaces in paths, portable
        args = [sys.executable, 'scripts/train_u1.py',
                '--data', r['data'],
                '--variant', r['variant'],
                '--seeds', str(r['seed']),
                '--hidden_dim', str(r.get('H', 64)),
                '--n_mp_blocks', str(r.get('B', 3)),
                '--epochs', str(r.get('epochs', 150)),
                '--run_prefix', r['prefix']]
        if r.get('n_train') is not None:
            args += ['--n_train', str(r['n_train'])]
        if r.get('gauge_augment'):
            args += ['--gauge_augment']
        print(f'[{i}/{len(runs)}] RUN: {rid}', flush=True)
        rc = subprocess.run(args).returncode
        if rc != 0:
            failures.append(rid)
            print(f'  FAILED rc={rc}')
    print(f'\n{len(runs) - len(failures)}/{len(runs)} ok')
    if failures:
        print('failed:', failures)


## F1a — augmented arms (the experiment)

Variants A and B with `--gauge_augment`, across the full n_train grid.
Includes n=3200: the direct augmented-vs-a4null comparison at matched
data budget.


In [ ]:
runs = [
    dict(prefix='a5aug', variant=v, data=DATA(8, 2.0), seed=s,
         n_train=n, gauge_augment=True)
    for v in ('link_nodes', 'edge_features')
    for n in SIZES
    for s in (0, 1, 2)
]
launch(runs)


## F1b — baseline arms (no augmentation)

Same grid WITHOUT augmentation — the control that separates
"augmentation helps" from "more data helps". n=3200 is skipped: that
point IS a4null (protocols coincide; the plot script reuses it).


In [ ]:
runs = [
    dict(prefix='a5base', variant=v, data=DATA(8, 2.0), seed=s, n_train=n)
    for v in ('link_nodes', 'edge_features')
    for n in SIZES if n != 3200
    for s in (0, 1, 2)
]
launch(runs)


## F1c — Variant C reference (the ceiling)

The invariant-input oracle on the same size grid: the data-efficiency
ceiling the augmented A/B curves are measured against. No augmentation —
C's inputs are bit-identical under gauge transforms anyway (that IS the
oracle). n=3200 is skipped (= a4C seeds at this cell).


In [ ]:
runs = [
    dict(prefix='a5C', variant='invariant_oracle', data=DATA(8, 2.0), seed=s,
         n_train=n)
    for n in SIZES if n != 3200
    for s in (0, 1, 2)
]
launch(runs)


## F2 — (optional) volume check

Does the augmentation effect (whatever it is at L=8) survive at L=16?
Full-size augmented A/B at the a4null L=16 beta=2 cell; compare against
the existing a4null L16 records.


In [ ]:
runs = [
    dict(prefix='a5aug', variant=v, data=DATA(16, 2.0), seed=s,
         n_train=3200, gauge_augment=True)
    for v in ('link_nodes', 'edge_features')
    for s in (0, 1, 2)
]
launch(runs)


## F3 — eps_gauge on the new checkpoints

The committed measurement script (same one that produced the A-4
checkpoint table on the laptop): K=32 gauge copies per test config,
per-config seeded, batch-1 forwards (batched evaluation has a ~1e-8
position-in-batch artifact that would spoil the Variant C exact zero —
see the script docstring).

CPU-bound (~2-4 min per run x 102 runs) — run it here, or on the laptop
after Drive sync; the skip logic makes it resumable either way. If a
session is short, `--max_configs 100` quarters the time (document that
in any figure caption that uses the reduced protocol).


In [ ]:
!python scripts/measure_gauge_invariance.py --runs "a5aug_*" "a5base_*" "a5C_*"


## Aggregate: eps_gauge table + augmentation curves

Committed deliverable scripts (single implementation, laptop and Colab).
`make_a5_table.py` aggregates every a5eps record; `plot_a5_curves.py`
draws accuracy + eps_gauge vs n_train, merging the a4null/a4C records
for the full-size no-aug points.


In [ ]:
!python scripts/make_a5_table.py
!python scripts/plot_a5_curves.py


## Done

Sync check: confirm `results/a5*.json`, `figures/a5_augmentation.*`, and
`experiments/runs/u1/a5*` show up on the laptop's Drive mirror. Claude
Code picks these up for the A-5 eps_gauge table, the augmentation-curve
figure, and the half-page paper summary (written once these results are
in).
